# Il costo del coordinamento

Il codice del capitolo [«Il costo del coordinamento»](https://book.paithon.it/main/SistemiMultiAgente/costo-del-coordinamento.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q numpy torch torchvision

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## Il costo del coordinamento

[Leggi la pagina](https://book.paithon.it/main/SistemiMultiAgente/costo-del-coordinamento.html)


### Il conto dei token


In [ ]:
# Costo in token di una conversazione a trascrizione condivisa:
# a ogni turno chi parla rilegge tutto cio' che e' stato detto finora.

def token_letti(turni, contesto_iniziale, messaggio_medio):
    """Token in ingresso sommati su tutti i turni: al turno t la finestra
    contiene il contesto iniziale piu' i t messaggi in trascrizione
    (l'enunciato del compito e i t-1 interventi precedenti)."""
    return sum(contesto_iniziale + t * messaggio_medio
               for t in range(1, turni + 1))


c0 = 2000   # istruzioni di sistema e definizione dei ruoli
m = 500     # lunghezza media di un messaggio
giri = 8    # giri di parola a testa

base = token_letti(giri, c0, m)  # un agente solo: il termine di paragone

print("agenti  turni  token letti  finestra finale  costo")
for agenti in (1, 2, 4, 8):
    turni = agenti * giri
    letti = token_letti(turni, c0, m)
    finestra = c0 + turni * m    # quanto e' larga la finestra all'ultimo turno
    print(f"{agenti:6d} {turni:6d} {letti:12,d} {finestra:16,d} {letti / base:6.1f}x")

## Chi parla con chi: le topologie del coordinamento

[Leggi la pagina](https://book.paithon.it/main/SistemiMultiAgente/topologie.html)


### La gerarchia


In [ ]:
import numpy as np

INF = 10**6


def distanze(A):
    """Cammini minimi fra tutte le coppie (Floyd-Warshall), grafo non pesato."""
    D = np.where(A == 1, 1, INF)
    np.fill_diagonal(D, 0)
    for k in range(len(A)):
        D = np.minimum(D, D[:, k, None] + D[None, k, :])
    return D


def grafo(n, archi):
    A = np.zeros((n, n), dtype=int)
    for i, j in archi:
        A[i, j] = A[j, i] = 1
    return A


def metriche(A, agenti):
    """Diametro fra agenti, grado massimo, carico del nodo piu' sollecitato
    (frazione di cammini minimi che lo attraversano) e frazione di coppie
    ancora collegate dopo averlo tolto."""
    D = distanze(A)
    coppie = [(s, t) for i, s in enumerate(agenti) for t in agenti[i + 1:]]
    carico = [sum(1 for s, t in coppie
                  if v not in (s, t) and D[s, v] + D[v, t] == D[s, t]) / len(coppie)
              for v in range(len(A))]
    v = int(np.argmax(carico))                       # il nodo piu' sollecitato
    Dv = distanze(np.delete(np.delete(A, v, 0), v, 1))
    resta = [a - (a > v) for a in agenti if a != v]   # indici dopo la rimozione
    vive = sum(1 for i, s in enumerate(resta) for t in resta[i + 1:]
               if Dv[s, t] < INF)
    n = len(resta)
    return (max(D[s, t] for s, t in coppie), A.sum(axis=1).max(),
            carico[v], vive / (n * (n - 1) / 2))


N = 15  # quindici agenti in tutte le topologie ad albero: stessa squadra, forme diverse
stella = grafo(N, [(0, i) for i in range(1, N)])
catena = grafo(N, [(i, i + 1) for i in range(N - 1)])
albero = grafo(N, [(i, 2*i + 1) for i in range(7)] + [(i, 2*i + 2) for i in range(7)])
lavagna = grafo(N + 1, [(N, i) for i in range(N)])   # il nodo N e' la lavagna

# La terza rete di Baran, la maglia: ipercubo a 4 dimensioni. Sedici agenti,
# arco fra due numeri che in binario differiscono per una sola cifra.
ipercubo = grafo(16, [(i, i ^ (1 << b))
                      for i in range(16) for b in range(4) if i < (i ^ (1 << b))])

print(f"{'topologia':10} {'diametro':>9} {'grado max':>10} {'carico':>7} {'residua':>8}")
for nome, A, agenti in [("stella", stella, list(range(N))),
                        ("catena", catena, list(range(N))),
                        ("albero", albero, list(range(N))),
                        ("lavagna", lavagna, list(range(N))),
                        ("ipercubo", ipercubo, list(range(16)))]:
    d, g, c, r = metriche(A, agenti)
    print(f"{nome:10} {d:9d} {g:10d} {c:6.0%} {r:7.0%}")

# La tabella stampa solo il nodo piu' carico. Nell'albero conviene guardarli
# tutti, perche' il primo della classe non e' quello che ci si aspetta.
D = distanze(albero)
coppie = [(s, t) for s in range(N) for t in range(s + 1, N)]
carico = [sum(1 for s, t in coppie
              if v not in (s, t) and D[s, v] + D[v, t] == D[s, t]) / len(coppie)
          for v in range(N)]
print(f"\nalbero: radice {carico[0]:.0%}, capo intermedio {carico[1]:.0%}, "
      f"foglia {carico[7]:.0%}")

## Mettersi d'accordo: dire, votare, diffidare

[Leggi la pagina](https://book.paithon.it/main/SistemiMultiAgente/protocolli-e-consenso.html)


### Un messaggio è una mossa


In [ ]:
from dataclasses import dataclass


@dataclass
class Messaggio:
    """La performativa dice che cosa il messaggio *fa*, non di che cosa parla."""
    performativa: str   # richiedi, accetta, rifiuta, informa, fallisci
    mittente: str
    destinatario: str
    filo: str           # a quale conversazione appartiene
    contenuto: str


# La macchina a stati del protocollo: da ogni stato, quali performative sono
# lecite e in quale stato portano.
TRANSIZIONI = {
    "aperto":    {"richiedi": "in attesa"},
    "in attesa": {"accetta": "impegnato", "rifiuta": "chiuso"},
    "impegnato": {"informa": "chiuso", "fallisci": "chiuso"},
    "chiuso":    {},
}

# Gli stati in cui la conversazione non e' finita: sono i conti aperti.
IN_SOSPESO = {"in attesa": "<- richiesta senza risposta",
              "impegnato": "<- impegno non onorato"}


def ripercorri(traccia):
    """Rilegge la traccia e restituisce lo stato finale di ogni filo.
    Solleva un errore alla prima mossa che il protocollo non prevede."""
    stati = {}
    for m in traccia:
        stato = stati.get(m.filo, "aperto")
        lecite = TRANSIZIONI[stato]
        if m.performativa not in lecite:
            raise ValueError(
                f"filo {m.filo}: '{m.performativa}' non e' lecita nello stato "
                f"'{stato}' (attese: {sorted(lecite) or 'nessuna'})")
        stati[m.filo] = lecite[m.performativa]
    return stati


traccia = [
    Messaggio("richiedi", "pianificatore", "ricercatore", "f1", "trova i dati"),
    Messaggio("accetta",  "ricercatore", "pianificatore", "f1", "procedo"),
    Messaggio("richiedi", "pianificatore", "analista", "f2", "stima il trend"),
    Messaggio("informa",  "ricercatore", "pianificatore", "f1", "ecco la tabella"),
    Messaggio("accetta",  "analista", "pianificatore", "f2", "procedo"),
    Messaggio("richiedi", "pianificatore", "revisore", "f3", "controlla la stima"),
    Messaggio("rifiuta",  "revisore", "pianificatore", "f3", "mi manca la tabella"),
]

for filo, stato in sorted(ripercorri(traccia).items()):
    print(f"{filo}: {stato:10s} {IN_SOSPESO.get(stato, '')}".rstrip())

# Una mossa fuori protocollo viene intercettata subito.
fuori = Messaggio("informa", "revisore", "pianificatore", "f3", "ecco il controllo")
try:
    ripercorri(traccia + [fuori])
except ValueError as errore:
    print("violazione:", errore)

### L'ipotesi che non regge


In [ ]:
from math import comb


def maggioranza(n, p):
    """Probabilita' che piu' della meta' di n votanti indipendenti sia corretta."""
    return sum(comb(n, k) * p**k * (1 - p) ** (n - k)
               for k in range(n // 2 + 1, n + 1))


def maggioranza_correlata(n, p0, lam):
    """Con probabilita' lam la domanda e' una trappola e sbagliano tutti insieme;
    altrimenti ciascuno e' corretto in modo indipendente con probabilita' p0."""
    return (1 - lam) * maggioranza(n, p0)


lam, p0 = 0.2, 0.875   # cosi' il singolo agente resta corretto il 70% delle volte

print("  n   indipendenti   correlati")
for n in (1, 3, 5, 9, 21, 99):
    print(f"{n:3d}       {maggioranza(n, 0.7):.3f}         "
          f"{maggioranza_correlata(n, p0, lam):.3f}")

# Nove agenti che rispondono all'unisono: quanto vale quell'unanimita'?
concordi_giusti = (1 - lam) * p0**9
concordi_sbagliati = lam           # sulle trappole sbagliano tutti allo stesso modo
unanimi = concordi_giusti + concordi_sbagliati
print(f"\nP(unanimita' su 9) = {unanimi:.3f}, "
      f"di cui sbagliate {concordi_sbagliati / unanimi:.1%}")

## Imparare insieme: quando l'ambiente impara anche lui

[Leggi la pagina](https://book.paithon.it/main/SistemiMultiAgente/imparare-insieme.html)


### Quando la scala non esiste


In [ ]:
import numpy as np

# Gioco ciclico a somma zero. A[i, j] e' il guadagno di chi gioca i contro
# chi gioca j: +1 vittoria, -1 sconfitta, 0 pareggio.
#             sasso  carta  forbici
A = np.array([[  0,   -1,    +1],    # sasso
              [ +1,    0,    -1],    # carta
              [ -1,   +1,     0]])   # forbici
NOMI = ["sasso", "carta", "forbici"]


def miglior_risposta(q):
    """L'azione che rende di piu' contro un avversario distribuito come q."""
    return int(np.argmax(A @ q))


def pura(i):
    """La distribuzione concentrata su una sola azione."""
    e = np.zeros(3)
    e[i] = 1.0
    return e


# Self-play ingenuo: ogni versione e' la miglior risposta all'ULTIMA versione.
storia = [0, 1]                      # gen 1 gioca sasso, gen 2 e' la sua risposta
print("gen  gioca     vs gen-1  vs gen-2  vs le passate  sfruttabilita'")
for t in range(3, 9):
    nuova = miglior_risposta(pura(storia[-1]))
    popolazione = np.mean([pura(p) for p in storia], axis=0)
    print(f"{t:3d}  {NOMI[nuova]:9s} {A[nuova, storia[-1]]:+7.2f} "
          f"{A[nuova, storia[-2]]:+8.2f} {(A @ popolazione)[nuova]:+13.2f} "
          f"{np.max(A @ pura(nuova)):+14.2f}")
    storia.append(nuova)

# Contro una POPOLAZIONE: miglior risposta alla media di tutte le versioni.
freq = np.array([1.0, 0.0, 0.0])
for _ in range(2000):
    freq[miglior_risposta(freq / freq.sum())] += 1
p = freq / freq.sum()
print("\npopolazione dopo 2000 generazioni: "
      + " ".join(f"{n}={v:.3f}" for n, v in zip(NOMI, p)))
print(f"sfruttabilita' della popolazione:  {np.max(A @ p):+.3f}")

### Il critico che vede tutto


In [ ]:
import torch
from torch import nn


class Attore(nn.Module):
    """Decentralizzato: vede solo la propria osservazione, in addestramento
    come in esecuzione. E' l'unica parte che sopravvive alla fine."""

    def __init__(self, dim_oss, dim_azione):
        super().__init__()
        self.rete = nn.Sequential(
            nn.Linear(dim_oss, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, dim_azione), nn.Tanh(),   # azioni continue in [-1, 1]
        )

    def forward(self, oss):
        return self.rete(oss)


class CriticoCentralizzato(nn.Module):
    """Esiste solo in addestramento: riceve osservazioni e azioni di TUTTI,
    cosi' la transizione che vede non dipende piu' dalle policy altrui.
    (Il bersaglio da regredire, invece, dipende ancora dal futuro: per
    quello servono comunque le reti target.)"""

    def __init__(self, dim_oss, dim_azione, n_agenti):
        super().__init__()
        ingresso = n_agenti * (dim_oss + dim_azione)
        self.rete = nn.Sequential(
            nn.Linear(ingresso, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, 1),                      # un solo numero: Q(x, a1..aN)
        )

    def forward(self, osservazioni, azioni):
        # due liste di N tensori (lotto, dim): si concatenano sull'asse features
        return self.rete(torch.cat(osservazioni + azioni, dim=1))


N, DIM_OSS, DIM_AZ, LOTTO = 3, 10, 2, 4

attori = nn.ModuleList([Attore(DIM_OSS, DIM_AZ) for _ in range(N)])
# un critico per agente: serve appena le ricompense non coincidono
critici = nn.ModuleList([CriticoCentralizzato(DIM_OSS, DIM_AZ, N) for _ in range(N)])

oss = [torch.randn(LOTTO, DIM_OSS) for _ in range(N)]   # cosa vede ciascuno
azioni = [attori[i](oss[i]) for i in range(N)]          # ognuno decide da solo

print(azioni[0].shape)             # torch.Size([4, 2])
print(critici[0](oss, azioni).shape)  # torch.Size([4, 1])

## Molti semplici invece di pochi intelligenti

[Leggi la pagina](https://book.paithon.it/main/SistemiMultiAgente/sciami-e-simulazioni.html)


### Rimescolare invece di muoversi: gli algoritmi genetici


In [ ]:
import numpy as np
from itertools import product

# --- l'istanza: 20 oggetti, uno zaino che regge 60 kg (fissata una volta) ---
istanza = np.random.default_rng(7)
N, CAPIENZA = 20, 60
peso   = istanza.integers(4, 20, N)
valore = istanza.integers(5, 40, N)

def bonta(pop):                      # quanto vale uno zaino; 0 se sfonda il limite
    return np.where(pop @ peso <= CAPIENZA, pop @ valore, 0)

def genetico(seme, POP=60, GEN=80, P_MUT=0.03):
    rng = np.random.default_rng(seme)
    pop = rng.integers(0, 2, size=(POP, N))          # una popolazione di zaini a caso
    for _ in range(GEN):
        f = bonta(pop)
        elite = pop[f.argmax()].copy()               # il migliore non si perde mai
        s = rng.integers(0, POP, size=(POP, 2))      # selezione: torneo a due
        genitori = np.where((f[s[:, 0]] >= f[s[:, 1]])[:, None], pop[s[:, 0]], pop[s[:, 1]])
        taglio = rng.integers(1, N, size=(POP, 1))   # incrocio a un punto:
        maschera = np.arange(N)[None, :] < taglio    # meta' da un genitore, meta' dall'altro
        figli = np.where(maschera, genitori, genitori[rng.permutation(POP)])
        figli ^= (rng.random((POP, N)) < P_MUT)      # mutazione: qualche bit ribaltato
        figli[0] = elite
        pop = figli
    return int(bonta(pop).max())

esiti = [genetico(s) for s in range(10)]
ottimo = max(sum(v for v, b in zip(valore, c) if b)
             for c in product([0, 1], repeat=N)
             if sum(p for p, b in zip(peso, c) if b) <= CAPIENZA)

print("dieci esecuzioni:", esiti)
print("ottimo vero (forza bruta su 2^20 = 1 048 576 combinazioni):", ottimo)
print(f"quante volte lo trova: {esiti.count(ottimo)}/10, con 4800 zaini provati su un milione")

### Uno sciame in venti righe


In [ ]:
import numpy as np


# Rastrigin in due dimensioni: minimo globale in (0, 0), dove vale 0.
# Tutt'attorno un reticolo di conche locali, attorno a ogni coppia di interi.
def rastrigin(X):
    return 10 * X.shape[1] + np.sum(X**2 - 10 * np.cos(2 * np.pi * X), axis=1)


rng = np.random.default_rng(7)     # seme fisso: il risultato e' riproducibile
n, d = 30, 2                       # trenta particelle in due dimensioni
w, c1, c2 = 0.73, 1.50, 1.50       # inerzia, spinta personale, spinta sociale

X = rng.uniform(-5.12, 5.12, (n, d))       # posizioni iniziali, sparse a caso
V = rng.uniform(-1.0, 1.0, (n, d))         # velocita' iniziali
P, fP = X.copy(), rastrigin(X)             # miglior punto di ogni particella
g = int(np.argmin(fP))                     # indice del migliore del gruppo

for t in range(1, 61):
    r1, r2 = rng.random((n, d)), rng.random((n, d))
    V = w * V + c1 * r1 * (P - X) + c2 * r2 * (P[g] - X)
    X = np.clip(X + V, -5.12, 5.12)        # nessuno esce dal dominio
    f = rastrigin(X)
    meglio = f < fP                        # chi ha battuto il proprio record
    P[meglio], fP[meglio] = X[meglio], f[meglio]
    g = int(np.argmin(fP))
    if t % 10 == 0:
        print(f"iterazione {t:3d}   f = {fP[g]:.6f}   "
              f"x = ({P[g][0]:+.4f}, {P[g][1]:+.4f})")

### Il confronto che si legge in giro, e quello onesto


In [ ]:
# Le trecento prove del confronto: semi 0..299. Ogni prova usa gli STESSI
# trenta punti iniziali per lo sciame e per le trenta ripartenze della
# discesa; la rastrigin e' quella definita sopra.
def sciame(X0, rng):
    n, d = X0.shape
    w, c1, c2 = 0.73, 1.50, 1.50
    X = X0.copy()
    V = rng.uniform(-1.0, 1.0, (n, d))
    P, fP = X.copy(), rastrigin(X)
    g = int(np.argmin(fP))
    for t in range(1, 61):
        r1, r2 = rng.random((n, d)), rng.random((n, d))
        V = w * V + c1 * r1 * (P - X) + c2 * r2 * (P[g] - X)
        X = np.clip(X + V, -5.12, 5.12)
        f = rastrigin(X)
        meglio = f < fP
        P[meglio], fP[meglio] = X[meglio], f[meglio]
        g = int(np.argmin(fP))
    return fP[g]


def discese(X0, passo=0.005, passi=2000):
    X = X0.copy()
    for _ in range(passi):
        G = 2 * X + 20 * np.pi * np.sin(2 * np.pi * X)   # gradiente esatto
        X = np.clip(X - passo * G, -5.12, 5.12)
    return rastrigin(X)


s_ok = d1_ok = d30_ok = singole_ok = conca = conca_ok = 0
for seme in range(300):
    rng = np.random.default_rng(seme)
    X0 = rng.uniform(-5.12, 5.12, (30, 2))
    f_sciame, f_disc = sciame(X0, rng), discese(X0)
    s_ok += f_sciame < 1e-2                   # la soglia del «fondo vero»
    d1_ok += f_disc[0] < 1e-2                 # partenza singola: il primo punto
    d30_ok += f_disc.min() < 1e-2             # il migliore delle trenta
    singole_ok += int((f_disc < 1e-2).sum())  # tutte le 9000 partenze singole
    in_conca = np.all(np.abs(X0) < 0.5, axis=1).any()
    conca += in_conca
    conca_ok += in_conca and (f_disc.min() < 1e-2)

p = singole_ok / 9000
print(f"sciame:                       {s_ok} su 300")
print(f"discesa, partenza singola:    {d1_ok} su 300")
print(f"discesa, trenta ripartenze:   {d30_ok} su 300")
print(f"partenze singole riuscite:    {singole_ok} su 9000")
print(f"prove nate in conca centrale: {conca} (riuscite: {conca_ok})")
print(f"atteso dal conto dei biglietti: {round((1 - (1 - p) ** 30) * 300)} su 300")